# Trace figure: demo on the running example

Draft for the visit conceptualisation (`overview/Single Level Visit Mappings.md`); the figure is not part of the tool yet.
The functions live in `ocelescope_module_location_based_ocpm.visits.trace_figure`; their documentation is in `docs/resources/trace-figure.md` and in the docstrings (e.g. `help(trace_figure)`).

Start Jupyter from the repository root with

```
uv run --with jupyter jupyter lab notebooks/trace_figure.ipynb
```

In [ ]:
import json
from pathlib import Path

import polars as pl
from ocelescope import OCEL

from ocelescope_module_location_based_ocpm.derivation.config import DerivationConfig
from ocelescope_module_location_based_ocpm.derivation.pipeline import derive
from ocelescope_module_location_based_ocpm.visits.trace_figure import (
    add_high_level_locations,
    add_location_kinds,
    high_level_runs,
    relations_for_figure,
    trace_figure,
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXAMPLE = ROOT / "data" / "running_example"

## 1. Derive the locations

The running example log and its configuration (`data/running_example/configuration.json`, Level 3).
`derive` returns a `Derivation`; its `relations` hold one row per hu-event relation with $loc_0$ to $loc_3$, and `derivation.loc` names the column of the selected level.

In [ ]:
ocel = OCEL.read(EXAMPLE / "running_example.sqlite")
config = DerivationConfig.model_validate(json.loads((EXAMPLE / "configuration.json").read_text()))
derivation = derive(ocel, config)
derivation.loc, derivation.relations.height

## 2. Prepare the relations: `relations_for_figure`

`relations_for_figure(derivation, handling_units, agg, vehicles, storage_units, axis)` selects the relations and adds everything the figure needs.

| parameter | meaning | default |
|---|---|---|
| `handling_units` | handling units to show, top to bottom in this order | all, sorted by id |
| `agg` | location mapping $agg: L \not\to HL$ as a dict; locations it does not map are their own high-level location (identity) | identity for all |
| `vehicles` | the set $V$ (drawn as open circles) | empty |
| `storage_units` | the set $W$ (drawn as diamonds); a location in both $V$ and $W$ counts as a vehicle | empty |
| `axis` | x axis: `"event order"` (shared events line up), `"trace position"` (index in $otrace(hu)$, as in the visit definition) or `"time"` | `"event order"` |

Here: the aggregation suggested for the running example (T is not mapped, so it stays T).

In [ ]:
AGG = {"A1": "A", "A2": "A", "B1": "B"}
VEHICLES = {"T"}
STORAGE_UNITS = {"B1"}
HUS = ["s1", "c1", "c2", "c3", "c4", "c5"]

relations = relations_for_figure(derivation, HUS, AGG, VEHICLES, STORAGE_UNITS)
relations.filter(pl.col("hu") == "c1")

### The steps inside

`relations_for_figure` only chains three small functions, which also work on any frame with the right columns:
- `add_high_level_locations(frame, location, agg)`: column `high_level` $= \widehat{hl}$ (⊥ stays ⊥),
- `add_location_kinds(frame, location, vehicles, storage_units)`: column `location_kind` (`vehicle`, `storage`, `other`; ⊥ for ⊥),
- `add_x_positions(frame, axis)`: columns `position` (trace position) and `x`.

`high_level_runs(relations)` gives the runs of consecutive relations with the same high-level location (used by the ribbons).

In [ ]:
toy = pl.DataFrame({"loc": ["A1", "A2", "B1", "T", None]})
toy = add_high_level_locations(toy, "loc", AGG)
add_location_kinds(toy, "loc", VEHICLES, STORAGE_UNITS)

In [ ]:
high_level_runs(relations.filter(pl.col("hu") == "c1"))

## 3. The figure: `trace_figure`

| parameter | meaning | default |
|---|---|---|
| `relations` | output of `relations_for_figure` | |
| `visits` | optional table of visits, one row per visit (section 5) | none |
| `filled_groups` | visit groups drawn as filled bars; all others outlined | `("resolved",)` |
| `high_level_style` | how $\widehat{hl}$ is drawn: `"ribbon"`, `"squares"`, `"shading"`, `"outline"` (section 6) | `"ribbon"` |
| `show_event_labels` | event ids at an angle above the markers | `True` |
| `link_shared_events` | dashed lines joining an event shared by several handling units (ignored on the trace position axis) | `True` |
| `title` | figure title | none |
| `width` | figure width in pixels; also decides which ribbon labels fit (section 7) | 1200 |

Hover over a marker or ribbon for the event, activity, time, trace position and both locations.

In [ ]:
trace_figure(relations, title="Running example, Level 3").show()

## 4. Axes

- `"event order"`: rank of the event among all shown events; an event shared by several handling units has one x, and dashed lines join them.
- `"trace position"`: the index in each handling unit's own trace, so shared events have different x values (no dashed lines). This is the position the visit definition counts in.
- `"time"`: real time; gaps between events become visible.

In [ ]:
for axis in ("trace position", "time"):
    rel = relations_for_figure(derivation, HUS, AGG, VEHICLES, STORAGE_UNITS, axis=axis)
    trace_figure(rel, title=f"x = {axis}").show()

## 5. Visits

Visits are not computed yet. `visits` takes a table with the columns `hu`, `high_level` (location of the visit), `first_event`, `last_event` and `group` (e.g. "maximal", "resolved").
Each group gets its own block of lanes below the handling unit, labelled with the group name; visits sharing relations go into different lanes (`visit_lanes`), so contained and overlapping visits are visible.

Below: the maximal visits at κ = 2, written by hand for illustration.

In [ ]:
VISITS_K2 = pl.DataFrame(
    [
        ("s1", "A", "e1", "e3"),
        ("c1", "A", "e3", "e11"),
        ("c1", "B", "e7", "e7"),
        ("c1", "T", "e8", "e14"),
        ("c2", "A", "e3", "e11"),
        ("c2", "T", "e8", "e8"),
        ("c2", "B", "e12", "e15"),
        ("c3", "T", "e8", "e8"),
        ("c3", "B", "e10", "e14"),
        ("c4", "B", "e15", "e16"),
    ],
    schema=["hu", "high_level", "first_event", "last_event"],
    orient="row",
).with_columns(pl.lit("maximal").alias("group"))

trace_figure(relations, VISITS_K2, title="Maximal visits at κ = 2 (illustration)").show()

`filled_groups` decides which groups are filled; here the same visits as a filled group, as resolved visits will look later.

In [ ]:
trace_figure(relations, VISITS_K2, filled_groups=("maximal",), title="Filled visit bars").show()

## 6. High-level styles

- `"ribbon"` (default): one labelled bar per run of the same high-level location, in a row below the trace.
- `"squares"`: one square per relation in a row below the trace.
- `"shading"`: a tinted, labelled box per run behind the trace (no extra row).
- `"outline"`: a ring in the high-level colour around each location marker (no extra row).

In [ ]:
for style in ("squares", "shading", "outline"):
    trace_figure(relations, VISITS_K2, high_level_style=style, title=f"high_level_style = {style!r}").show()

## 7. Ribbon labels and `width`

A run label is only shown if it fits into its ribbon (or box); otherwise the legend and the hover name the location.
Plotly in Python does not know the rendered text width, so the fit is estimated for the figure width `width` (and the figure is drawn at that width).
With long names, a narrower figure shows fewer labels:

In [ ]:
LONG = {"A1": "Bay area A", "A2": "Bay area A", "B1": "Storage hall north", "T": "Truck"}
rel = relations_for_figure(derivation, ["c1", "c2", "c3"], LONG, VEHICLES, STORAGE_UNITS)
for width in (1600, 800):
    trace_figure(rel, title=f"width = {width}", width=width).show()

## 8. Labels and links off

For many handling units or long traces, the event ids and the shared-event lines can be switched off.

In [ ]:
trace_figure(relations, show_event_labels=False, link_shared_events=False, title="Without event ids and links").show()

## 9. Saving

`fig.write_html(path)` saves an interactive file (`include_plotlyjs="cdn"` gives a small file that needs internet to open).

In [ ]:
# trace_figure(relations, VISITS_K2).write_html(ROOT / "notebooks" / "trace-figure.html")